# Data Preprocessing

Turns the cleaned dataset into numeric feature vectors for training, using the project's own enums (`PersonHomeOwnership`, `LoanIntent`, `LoanGrade`) to one-hot encode the categorical fields.

**Contents:**
- Build feature vectors — numeric fields as-is, categorical fields one-hot encoded via their enum's `one_hot()`.
- Features and product rows — preview each row's feature vector next to its `loan_status` ("product") value.

In [ ]:
from src.fetch import fetch_dataset
from src.extraction import extract_data

df = extract_data(fetch_dataset())

## Build feature vectors

Each row becomes one numeric vector: the numeric fields as-is, followed by each categorical field's `one_hot()` encoding.

In [ ]:
import numpy as np
from src.primitives import LoanIntent, LoanGrade, PersonHomeOwnership, DefaultOnFile

NUMERIC_FEATURES = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
]


def to_feature_vector(row) -> np.ndarray:
    numeric = row[NUMERIC_FEATURES].to_numpy(dtype=float)
    home_ownership = PersonHomeOwnership(row["person_home_ownership"]).one_hot()
    intent = LoanIntent(row["loan_intent"]).one_hot()
    grade = LoanGrade(row["loan_grade"]).one_hot()
    default_on_file = DefaultOnFile(row["cb_person_default_on_file"]).encode_flag()

    return np.concatenate([numeric, home_ownership, intent, grade, default_on_file])


X = np.stack(df.apply(to_feature_vector, axis=1))
y = df["loan_status"].to_numpy()

X.shape, y.shape

## Features and product rows

Each row's encoded feature vector alongside its `loan_status` ("product") value.

**Column order** (25 values total, in the order they're concatenated in `to_feature_vector`):

| Range | Field | Width |
| --- | --- | --- |
| 0–6 | `NUMERIC_FEATURES` as-is (`person_age`, `person_income`, `person_emp_length`, `loan_amnt`, `loan_int_rate`, `loan_percent_income`, `cb_person_cred_hist_length`) | 7 |
| 7–10 | `person_home_ownership` one-hot, ordered per `PersonHomeOwnership.values()` | 4 |
| 11–16 | `loan_intent` one-hot, ordered per `LoanIntent.values()` | 6 |
| 17–23 | `loan_grade` one-hot, ordered per `LoanGrade.values()` | 7 |
| 24 | `cb_person_default_on_file`, encoded via `DefaultOnFile.encode_flag()` (1 = defaulted before) | 1 |

**Product** is `loan_status` — the target the model predicts, not a feature: `1` means the loan defaulted, `0` means it didn't.

In [ ]:
from IPython.display import display
import pandas as pd

preview = pd.DataFrame({"features": list(X[:5]), "product": y[:5]})
display(preview)